# FOCUS-3D Fine-tuning Demo

This notebook demonstrates how to fine-tune FOCUS-3D using user-curated 3D image-label pairs.

The workflow is:

1. Prepare paired 3D images and labels.
2. Crop them into training patches.
3. Save patches into `imagesTr/` and `labelsTr/`.
4. Fine-tune FOCUS-3D from an existing checkpoint.

Input labels should be 3D instance label maps:

- background = 0
- each cell instance = a positive integer ID

In [ ]:
# ============================================================
# Cell 1. Environment and imports
# ============================================================
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import tifffile

# ------------------------------------------------------------
# GPU setting
# ------------------------------------------------------------
# For single-GPU notebook training, use one GPU, for example "0".
# For multi-GPU command-line training, use "0,1,2,3".
#
# If you change CUDA_VISIBLE_DEVICES, restart the notebook kernel.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------
# This notebook is assumed to be located at:
#   focus3d/notebooks/02_finetune.ipynb
#
# Backend code is located at:
#   focus3d/src/focus3d/segmentation/FOCUS3D
# ------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
FOCUS3D_BACKEND_DIR = (
    PROJECT_ROOT / 'src' / 'focus3d' / 'segmentation' / 'FOCUS3D'
).resolve()

if str(FOCUS3D_BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(FOCUS3D_BACKEND_DIR))

from fine_tune_function import (  # noqa: E402
    cfg_value,
    check_exists,
    export_training_patches,
)
from train_net_3d import (  # noqa: E402
    build_cfg,
    run_train,
    run_train_distributed,
)

print('FOCUS-3D fine-tuning notebook is ready.')
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('Project root:', PROJECT_ROOT)
print('Backend path:', FOCUS3D_BACKEND_DIR)

## User settings

Modify the following paths and training parameters.

You need paired image and label volumes. The recommended folder structure is:

```text
example/finetune_data/
├── images/
│   ├── volume_001.tif
│   └── volume_002.tif
└── labels/
    ├── volume_001.tif
    └── volume_002.tif

In [ ]:
# ============================================================
# Cell 2. User-editable settings
# ============================================================

# ------------------------------------------------------------
# Input image/label folders
# ------------------------------------------------------------
SOURCE_IMAGE_DIR = PROJECT_ROOT / 'example' / 'finetune_data' / 'images'
SOURCE_LABEL_DIR = PROJECT_ROOT / 'example' / 'finetune_data' / 'labels'

# ------------------------------------------------------------
# Patch dataset output
# ------------------------------------------------------------
PATCH_DATA_DIR = PROJECT_ROOT / 'example' / 'finetune_patches'
PATCH_IMAGE_DIR = PATCH_DATA_DIR / 'imagesTr'
PATCH_LABEL_DIR = PATCH_DATA_DIR / 'labelsTr'

# If True, existing patch data will be removed and regenerated.
RECREATE_PATCHES = True

# If you already have imagesTr/labelsTr, set this to False.
RUN_PATCH_EXPORT = True

# ------------------------------------------------------------
# Model config and checkpoints
# ------------------------------------------------------------
CONFIG_FILE = FOCUS3D_BACKEND_DIR / 'configs' / '3d_test.yaml'

# Full segmentation checkpoint used as fine-tuning initialization.
# This is usually model_final.pth.
INIT_SEG_CHECKPOINT = (
    PROJECT_ROOT / 'example' / 'checkpoint' / 'model_final.pth'
)

# Fine-tuning output folder.
OUTPUT_DIR = PROJECT_ROOT / 'example' / 'finetune_output'

# Skip patches with too little annotated foreground.
MIN_LABEL_INSTANCES = 3

# Skip patches whose raw image is nearly empty.
# Set to None to disable raw-intensity filtering.
BACKGROUND_THRESHOLD = None

# Limit patches per volume.
# Set to None to keep all valid patches.
MAX_PATCHES_PER_VOLUME = None

RANDOM_SEED = 2026

# ------------------------------------------------------------
# Fine-tuning parameters
# ------------------------------------------------------------
NUM_GPUS = 1

# Total batch size across all GPUs.
# For example, if NUM_GPUS=4 and IMS_PER_BATCH=16,
# each GPU processes approximately 4 patches.
IMS_PER_BATCH = 4

BASE_LR = 5e-5
MAX_ITER = 1000
WARMUP_ITERS = 100
CHECKPOINT_PERIOD = 500
EVAL_PERIOD = 0

# Resume from OUTPUT_DIR if an existing checkpoint is found.
RESUME = False

# Use AMP mixed precision.
AMP_ENABLED = True

# ------------------------------------------------------------
# Print settings
# ------------------------------------------------------------
settings = {
    'SOURCE_IMAGE_DIR': str(SOURCE_IMAGE_DIR),
    'SOURCE_LABEL_DIR': str(SOURCE_LABEL_DIR),
    'PATCH_DATA_DIR': str(PATCH_DATA_DIR),
    'CONFIG_FILE': str(CONFIG_FILE),
    'INIT_SEG_CHECKPOINT': str(INIT_SEG_CHECKPOINT),
    'OUTPUT_DIR': str(OUTPUT_DIR),
    'MIN_LABEL_INSTANCES': MIN_LABEL_INSTANCES,
    'BACKGROUND_THRESHOLD': BACKGROUND_THRESHOLD,
    'MAX_PATCHES_PER_VOLUME': MAX_PATCHES_PER_VOLUME,
    'NUM_GPUS': NUM_GPUS,
    'IMS_PER_BATCH': IMS_PER_BATCH,
    'BASE_LR': BASE_LR,
    'MAX_ITER': MAX_ITER,
}

print(json.dumps(settings, indent=2))

In [ ]:
# ============================================================
# Cell 3. Check paths
# ============================================================
if RUN_PATCH_EXPORT:
    check_exists(SOURCE_IMAGE_DIR, 'Source image folder')
    check_exists(SOURCE_LABEL_DIR, 'Source label folder')

check_exists(CONFIG_FILE, 'Config file')
check_exists(INIT_SEG_CHECKPOINT, 'Initial segmentation checkpoint')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output folder:', OUTPUT_DIR)

## Patch extraction

This section converts full 3D volumes into fixed-size training patches.

The output structure will be:

```text
finetune_patches/
├── imagesTr/
│   ├── volume_001_z0000_y0000_x0000.tif
│   └── ...
└── labelsTr/
    ├── volume_001_z0000_y0000_x0000.tif
    └── ...

In [ ]:
# ============================================================
# Cell 4. Run patch extraction
# ============================================================
if RUN_PATCH_EXPORT:
    patch_summary = export_training_patches(
        source_image_dir=SOURCE_IMAGE_DIR,
        source_label_dir=SOURCE_LABEL_DIR,
        patch_image_dir=PATCH_IMAGE_DIR,
        patch_label_dir=PATCH_LABEL_DIR,
        min_label_instances=MIN_LABEL_INSTANCES,
        background_threshold=BACKGROUND_THRESHOLD,
        max_patches_per_volume=MAX_PATCHES_PER_VOLUME,
        recreate=RECREATE_PATCHES,
        seed=RANDOM_SEED,
    )

    print('\nPatch summary:')
    for item in patch_summary:
        print(
            f'- {Path(item["image"]).name}: '
            f'{item["saved_patches"]} / {item["all_patches"]} patches saved'
        )
else:
    print('Patch export skipped. Existing imagesTr/labelsTr will be used.')

if not PATCH_IMAGE_DIR.exists() or not PATCH_LABEL_DIR.exists():
    raise FileNotFoundError('imagesTr/labelsTr folders are missing.')

n_images = len(list(PATCH_IMAGE_DIR.glob('*.tif')))
n_labels = len(list(PATCH_LABEL_DIR.glob('*.tif')))

print('\nTraining patch folders:')
print('Images:', PATCH_IMAGE_DIR, 'count =', n_images)
print('Labels:', PATCH_LABEL_DIR, 'count =', n_labels)

if n_images == 0 or n_labels == 0:
    raise RuntimeError('No training patches found.')

if n_images != n_labels:
    raise RuntimeError(
        'The number of image patches and label patches is different.'
    )

## Preview training patches

This section shows one exported training patch and its label.

This is a quick sanity check before starting fine-tuning.

In [ ]:
# ============================================================
# Cell 5. Preview one training patch
# ============================================================
patch_files = sorted(PATCH_IMAGE_DIR.glob('*.tif'))

preview_image_path = patch_files[1000]
preview_label_path = PATCH_LABEL_DIR / preview_image_path.name

preview_image = tifffile.imread(str(preview_image_path))
preview_label = tifffile.imread(str(preview_label_path))

z = preview_image.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Image
axes[0].imshow(preview_image[z], cmap='gray')
axes[0].set_title(f'Image middle slice, Z={z}')
axes[0].axis('off')

# Label
axes[1].imshow(preview_label[z], cmap='nipy_spectral', interpolation='nearest')
axes[1].set_title(f'Label middle slice, Z={z}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print('Preview image:', preview_image_path)
print('Preview label:', preview_label_path)
print('Max label ID:', int(preview_label.max()))

## Build fine-tuning configuration

The original `3d_test.yaml` is used as the base config.

The notebook overrides the most important parameters:

- training image folder;
- training label folder;
- output folder;
- initial segmentation checkpoint;
- MAE pretrained checkpoint;
- patch size;
- batch size;
- learning rate;
- maximum iterations;
- checkpoint period.

For notebook use, single-GPU training is recommended. Multi-GPU training is usually more stable from the command line.

In [ ]:
# ============================================================
# Cell 6. Build config overrides
# ============================================================
OPTS = [
    # Dataset paths
    'DATASETS.IMAGE_DIR',
    cfg_value(PATCH_IMAGE_DIR),
    'DATASETS.LABEL_DIR',
    cfg_value(PATCH_LABEL_DIR),
    # Output folder
    'OUTPUT_DIR',
    cfg_value(OUTPUT_DIR),
    # Initial full segmentation checkpoint
    'MODEL.WEIGHTS',
    cfg_value(INIT_SEG_CHECKPOINT),
    # Training hyperparameters
    'SOLVER.IMS_PER_BATCH',
    cfg_value(IMS_PER_BATCH),
    'SOLVER.BASE_LR',
    cfg_value(BASE_LR),
    'SOLVER.MAX_ITER',
    cfg_value(MAX_ITER),
    'SOLVER.WARMUP_ITERS',
    cfg_value(WARMUP_ITERS),
    'SOLVER.CHECKPOINT_PERIOD',
    cfg_value(CHECKPOINT_PERIOD),
    'SOLVER.AMP.ENABLED',
    cfg_value(AMP_ENABLED),
    # Evaluation
    'TEST.EVAL_PERIOD',
    cfg_value(EVAL_PERIOD),
]

print('Config overrides:')
for k, v in zip(OPTS[0::2], OPTS[1::2], strict=False):
    print(f'{k}: {v}')

In [ ]:
# ============================================================
# Cell 7. Create config object
# ============================================================

cfg = build_cfg(str(CONFIG_FILE), OPTS)

print('Final training config:')
print('DATASETS.IMAGE_DIR:', cfg.DATASETS.IMAGE_DIR)
print('DATASETS.LABEL_DIR:', cfg.DATASETS.LABEL_DIR)
print('OUTPUT_DIR:', cfg.OUTPUT_DIR)
print('MODEL.WEIGHTS:', cfg.MODEL.WEIGHTS)
print('MODEL.BACKBONE.PRETRAINED:', cfg.MODEL.BACKBONE.PRETRAINED)
print('INPUT.IMAGE_SIZE:', cfg.INPUT.IMAGE_SIZE)
print('SOLVER.IMS_PER_BATCH:', cfg.SOLVER.IMS_PER_BATCH)
print('SOLVER.BASE_LR:', cfg.SOLVER.BASE_LR)
print('SOLVER.MAX_ITER:', cfg.SOLVER.MAX_ITER)
print('SOLVER.CHECKPOINT_PERIOD:', cfg.SOLVER.CHECKPOINT_PERIOD)
print('SOLVER.AMP.ENABLED:', cfg.SOLVER.AMP.ENABLED)

Path(cfg.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## Start fine-tuning

For most notebook users, use `NUM_GPUS = 1`.

For multi-GPU fine-tuning, command-line training is recommended:

```bash
CUDA_VISIBLE_DEVICES=0,1,2,3 python train_net_3d.py \
  --config-file ./configs/3d_test.yaml \
  --num-gpus 4 \
  DATASETS.IMAGE_DIR /path/to/imagesTr \
  DATASETS.LABEL_DIR /path/to/labelsTr \
  OUTPUT_DIR /path/to/output \
  MODEL.WEIGHTS /path/to/model_final.pth

In [ ]:
# ============================================================
# Cell 8. Start fine-tuning
# ============================================================
from detectron2.modeling import META_ARCH_REGISTRY

print('Registered MaskFormer:', META_ARCH_REGISTRY.get('MaskFormer'))
if NUM_GPUS == 1:
    print('Starting single-GPU fine-tuning inside notebook...')
    print(
        'If you changed CUDA_VISIBLE_DEVICES, restart the kernel before running this cell.'
    )
    train_result = run_train(cfg, resume=RESUME)
else:
    print('Starting distributed fine-tuning from notebook...')
    print('For large multi-GPU jobs, running from terminal is recommended.')
    train_result = run_train_distributed(
        config_file=str(CONFIG_FILE),
        num_gpus=int(NUM_GPUS),
        resume=RESUME,
        opts=OPTS,
        dist_url='auto',
    )

print('Fine-tuning finished.')
print('Training result:', train_result)

In [ ]:
# ============================================================
# Cell 9. Locate checkpoints
# ============================================================

output_dir = Path(OUTPUT_DIR)
checkpoints = sorted(output_dir.glob('*.pth'))

print('Output folder:', output_dir)
print(f'Found {len(checkpoints)} checkpoint file(s).')

for p in checkpoints:
    print(p)

final_ckpt = output_dir / 'model_final.pth'
if final_ckpt.exists():
    print('\nFinal checkpoint:')
    print(final_ckpt)
else:
    print(
        '\nmodel_final.pth was not found yet. Training may have stopped before final saving.'
    )